# ProofyX detector training - 3 models, one per laptop
Run this **same notebook on three laptops**; only `TASK` changes.

| Laptop | `TASK` | Model | Dataset it pulls | Output |
|---|---|---|---|---|
| 1 | `face` | EfficientNet-B4 @380 | `pujanpaudel/deepfake_face_classification` (DF40 test, 16k real + 16k fake) | `face.pth` |
| 2 | `scene` | CLIP ViT-B/16 + linear head | `TheKernel01/Tiny-GenImage` (real + 7 generators) | `scene.pth` |
| 3 | `document` | ConvNeXt-Tiny @384 | real invoices/receipts + synthetic tampering | `document.pth` |

**Setup:** Runtime > Change runtime type > **GPU (T4 is enough)**. Put `training.zip` in Drive at `MyDrive/proofyx/`.
Training resumes automatically if Colab disconnects. Non-commercial dataset licences: read the note at the bottom.

In [ ]:
TASK = "face"        # <-- face | scene | document   (change per laptop)
PER_CLASS = 6000     # images per class to use (more = better, slower). Start 6000.
QUICK_TEST = False   # True: tiny run (300/class, 1 epoch) to check everything works before the real run

CFG = {   # tuned per model: (epochs, batch, lr_head, backbone_lr_mult, freeze_epochs)
 "face":     dict(epochs=10, bs=16, lr=5e-4, mult=0.2,  freeze=1),
 "scene":    dict(epochs=8,  bs=64, lr=1e-3, mult=0.01, freeze=3),   # CLIP: mostly linear probe, gentle fine-tune
 "document": dict(epochs=12, bs=16, lr=3e-4, mult=0.2,  freeze=1),
}[TASK]
if QUICK_TEST: PER_CLASS, CFG["epochs"] = 300, 1
DATA = f"/content/data_{TASK}"
OUT  = f"/content/drive/MyDrive/proofyx/{TASK}.pth"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile
os.makedirs('/content/drive/MyDrive/proofyx', exist_ok=True)
zipfile.ZipFile('/content/drive/MyDrive/proofyx/training.zip').extractall('/content/code')
%cd /content/code
!pip -q install datasets transformers huggingface_hub
!nvidia-smi -L

## 1. Data (run the cell for your TASK)

In [ ]:
if TASK == "scene":
    !python training/get_data.py scene --out {DATA} --per_class {PER_CLASS}
elif TASK == "face":
    !python training/get_data.py face --out {DATA} --per_class {PER_CLASS}
else:
    # Real documents first, then synthetic tampering. ADD YOUR OWN real scans/invoices to /content/real_docs for best results.
    !python training/get_data.py docs --out /content/real_docs --per_class {PER_CLASS}
    !python training/make_doc_fakes.py --real /content/real_docs --out {DATA}

In [ ]:
import glob
for sp in ("train","val"):
    for c in ("real","fake"):
        print(sp, c, len(glob.glob(f"{DATA}/{sp}/{c}/*")))
# Look at a few samples to be sure labels are right
import random; from PIL import Image
for c in ("real","fake"):
    f=random.choice(glob.glob(f"{DATA}/train/{c}/*")); print(c, f); display(Image.open(f).resize((256,256)))

## 2. Train  (saves the best checkpoint to Drive after every improving epoch)

In [ ]:
!python training/train.py --task {TASK} --data {DATA} --out {OUT} --epochs {CFG['epochs']} --bs {CFG['bs']} --lr {CFG['lr']} --backbone_lr_mult {CFG['mult']} --freeze_epochs {CFG['freeze']} --workers 2

## 3. Check and download
Look at **val AUC** and the real/fake accuracy at the chosen threshold. These come from the *same dataset* you trained on,
so they are optimistic. Before trusting a model, test it on images from a **different source** (e.g. your own APM photos).

In [ ]:
import torch
ck = torch.load(OUT, map_location="cpu")
print({k: ck[k] for k in ("task","arch","img_size","threshold")}); print(ck["metrics"])
from google.colab import files
files.download(OUT)

### Licence note
`Tiny-GenImage` is CC-BY-NC-SA and the face set derives from DF40 (research terms). Fine for research and a demo; **before selling a product built on these weights, get legal advice** and plan to fine-tune on data you own (APM photos, consented samples) - the review queue produces exactly that.